# SCT Rank Sweep — SmolLM2-1.7B
**Spectral Compact Training** — Compression vs Convergence tradeoff

Dense baseline vs SCT at ranks 32, 64, 128, 256 on Alpaca fine-tuning.

| Rank | MLP Compression | Total Params (est.) |
|------|----------------|--------------------|
| 256  | 5.9×           | ~1.33B             |
| 128  | 11.7×          | ~983M              |
| 64   | 23.5×          | ~735M              |
| 32   | 46.9×          | ~527M              |

---
**Runtime:** A100 GPU — ~2.5 hours total

**Budget:** ~14.5 compute units

---

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch transformers datasets matplotlib

In [ ]:
# Cell 2: Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
# Cell 3: SpectralLinear implementation
import math
import torch
import torch.nn as nn


class SpectralLinear(nn.Module):
    """W = U diag(s) V^T with Stiefel QR retraction."""

    def __init__(self, in_features, out_features, rank=32):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = min(rank, min(in_features, out_features))

        U = torch.randn(in_features, self.rank) / math.sqrt(in_features)
        V = torch.randn(out_features, self.rank) / math.sqrt(out_features)
        Q_U, R_U = torch.linalg.qr(U)
        Q_V, R_V = torch.linalg.qr(V)
        self.U = nn.Parameter((Q_U * torch.sign(torch.diag(R_U)))[:, :self.rank])
        self.V = nn.Parameter((Q_V * torch.sign(torch.diag(R_V)))[:, :self.rank])
        self.s = nn.Parameter(torch.ones(self.rank))

    def forward(self, x):
        return (x @ self.U) * self.s @ self.V.T

    @classmethod
    def from_linear(cls, linear, rank=None, energy=None):
        """Convert nn.Linear to SpectralLinear via truncated SVD."""
        W = linear.weight.data.float()
        U_full, S_full, Vt_full = torch.linalg.svd(W, full_matrices=False)

        if energy is not None:
            total_energy = (S_full ** 2).sum()
            cumulative = torch.cumsum(S_full ** 2, dim=0) / total_energy
            k = max(1, int((cumulative >= energy).nonzero(as_tuple=True)[0][0].item()) + 1)
            if rank is not None:
                k = min(k, rank)
        elif rank is not None:
            k = min(rank, S_full.shape[0])
        else:
            k = min(32, S_full.shape[0])

        layer = cls.__new__(cls)
        nn.Module.__init__(layer)
        layer.in_features = linear.in_features
        layer.out_features = linear.out_features
        layer.rank = k
        layer.U = nn.Parameter(Vt_full[:k, :].T.contiguous())
        layer.V = nn.Parameter(U_full[:, :k].contiguous())
        layer.s = nn.Parameter(S_full[:k].contiguous())
        return layer

    @torch.no_grad()
    def retract(self):
        """QR retraction to Stiefel manifold."""
        for M in [self.U, self.V]:
            Q, R = torch.linalg.qr(M.data)
            M.data = (Q * torch.sign(torch.diag(R)))[:, :self.rank]


def retract_all(module):
    for m in module.modules():
        if isinstance(m, SpectralLinear):
            m.retract()


def convert_model_mlp_to_spectral(model, rank=None, energy=None):
    """Convert all MLP linear layers to SpectralLinear."""
    converted = 0
    for name, module in model.named_modules():
        for attr in ['gate_proj', 'up_proj', 'down_proj']:
            if hasattr(module, attr):
                linear = getattr(module, attr)
                if isinstance(linear, nn.Linear):
                    spectral = SpectralLinear.from_linear(linear, rank=rank, energy=energy)
                    setattr(module, attr, spectral)
                    converted += 1
    return converted


print("SpectralLinear ready.")

In [ ]:
# Cell 4: Load data
from datasets import load_dataset
from transformers import AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B")
tokenizer.pad_token = tokenizer.eos_token

print("Loading Alpaca dataset...")
ds = load_dataset("tatsu-lab/alpaca", split="train")

MAX_LENGTH = 512

def format_sample(ex):
    if ex.get("input", "").strip():
        return f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{ex['input']}\n\n### Response:\n{ex['output']}"
    return f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['output']}"

texts = [format_sample(ex) for ex in ds]
encodings = tokenizer(texts, truncation=True, max_length=MAX_LENGTH,
                      padding="max_length", return_tensors="pt")

print(f"Samples: {encodings['input_ids'].shape[0]}  Seq length: {MAX_LENGTH}")

In [ ]:
# Cell 5: Training function
import time
import json

def train(model, encodings, device, steps, lr, batch_size, log_every,
          is_sct=False, label=""):
    """Train and return loss history."""
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    input_ids = encodings["input_ids"]
    attention_mask = encodings["attention_mask"]
    n_samples = input_ids.shape[0]

    losses = []
    step_times = []

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*60}")
    print(f"  Training: {label}")
    print(f"  Steps: {steps}  LR: {lr}  Batch: {batch_size}")
    print(f"  Trainable params: {trainable:,}")
    print(f"{'='*60}\n")

    # Reset peak stats so we report this run's actual memory
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    for step in range(1, steps + 1):
        idx = torch.randint(0, n_samples, (batch_size,))
        batch_input = input_ids[idx].to(device)
        batch_mask = attention_mask[idx].to(device)
        labels = batch_input.clone()
        labels[batch_mask == 0] = -100

        t0 = time.time()
        optimizer.zero_grad()
        outputs = model(input_ids=batch_input, attention_mask=batch_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if is_sct:
            retract_all(model)

        step_time = time.time() - t0
        loss_val = loss.item()
        losses.append({"step": step, "loss": loss_val, "time": step_time})
        step_times.append(step_time)

        if step % log_every == 0 or step == 1:
            ppl = math.exp(min(loss_val, 20))
            avg_time = sum(step_times[-log_every:]) / len(step_times[-log_every:])
            eta_min = (steps - step) * avg_time / 60
            if torch.cuda.is_available():
                mem = torch.cuda.max_memory_allocated() / 1e9
                print(f"  [{label}] Step {step:>5d}/{steps}  "
                      f"Loss: {loss_val:.4f}  PPL: {ppl:.2f}  "
                      f"Step: {avg_time:.2f}s  GPU: {mem:.1f}GB  ETA: {eta_min:.1f}min")
            else:
                print(f"  [{label}] Step {step:>5d}/{steps}  "
                      f"Loss: {loss_val:.4f}  PPL: {ppl:.2f}  "
                      f"Step: {avg_time:.2f}s  ETA: {eta_min:.1f}min")

    return losses

print("Training function ready.")

In [ ]:
# Cell 6: Configuration
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

STEPS = 2000          # Training steps per method
DENSE_LR = 2e-5       # Dense fine-tuning LR
SCT_LR = 5e-4         # SCT needs higher LR to recover from truncation
BATCH_SIZE = 4        # Batch size
LOG_EVERY = 50        # Print every N steps
RANKS = [256, 128, 64, 32]  # Rank sweep from high to low
MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Config: {STEPS} steps, batch {BATCH_SIZE}")
print(f"Dense LR: {DENSE_LR}, SCT LR: {SCT_LR}")
print(f"Rank sweep: {RANKS}")
print(f"Device: {device}")

In [ ]:
# Cell 7: Dense baseline training
from transformers import AutoModelForCausalLM
import gc

print("Loading SmolLM2-1.7B for dense baseline...")
model_dense = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
).to(device)

dense_params = sum(p.numel() for p in model_dense.parameters())
print(f"Dense parameters: {dense_params:,}")

dense_losses = train(
    model_dense, encodings, device,
    steps=STEPS, lr=DENSE_LR, batch_size=BATCH_SIZE,
    log_every=LOG_EVERY, is_sct=False, label="Dense"
)

with open("dense_losses.json", "w") as f:
    json.dump(dense_losses, f)
print("Dense results saved.")

del model_dense
torch.cuda.empty_cache()
gc.collect()
print("Memory freed.")

In [ ]:
# Cell 8: SCT training
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoModelForCausalLM
import gc
import torch

print("Clearing memory...")
torch.cuda.empty_cache()
gc.collect()

print("Loading SmolLM2-1.7B for SCT conversion...")
model_sct = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
).to(device)

print(f"Converting MLP layers to SpectralLinear (rank={RANK})...")
n_converted = convert_model_mlp_to_spectral(model_sct, rank=RANK)
print(f"Converted {n_converted} layers")

# Match dtypes
for module in model_sct.modules():
    if isinstance(module, SpectralLinear):
        module.U.data = module.U.data.to(model_sct.dtype)
        module.V.data = module.V.data.to(model_sct.dtype)
        module.s.data = module.s.data.to(model_sct.dtype)

# Safe retract for bfloat16
@torch.no_grad()
def safe_retract(module):
    if isinstance(module, SpectralLinear):
        for M in [module.U, module.V]:
            original_dtype = M.data.dtype
            M_float = M.data.to(torch.float32)
            Q, R = torch.linalg.qr(M_float)
            M.data = (Q * torch.sign(torch.diag(R)))[:, :module.rank].to(original_dtype)

def safe_retract_all(model):
    for m in model.modules():
        safe_retract(m)

# Override the retract_all used by train()
retract_all = safe_retract_all

sct_params = sum(p.numel() for p in model_sct.parameters() if p.requires_grad)
print(f"SCT parameters: {sct_params:,}")

ranks = [m.rank for m in model_sct.modules() if isinstance(m, SpectralLinear)]
if ranks:
    print(f"Rank range: {min(ranks)}-{max(ranks)} (mean: {sum(ranks)/len(ranks):.0f})")

torch.cuda.reset_peak_memory_stats()

SCT_LR = 5e-4

sct_losses = train(
    model_sct, encodings, device,
    steps=STEPS, lr=SCT_LR, batch_size=BATCH_SIZE,
    log_every=LOG_EVERY, is_sct=True, label="SCT"
)

with open("sct_losses.json", "w") as f:
    json.dump(sct_losses, f)
print("SCT results saved.")

del model_sct
torch.cuda.empty_cache()

In [ ]:
# Cell 9: Generate rank sweep convergence plot
import matplotlib.pyplot as plt
import json
import math

# Load all results
with open("dense_losses.json") as f:
    dense_losses = json.load(f)

sct_data = {}
for rank in RANKS:
    with open(f"sct_r{rank}_losses.json") as f:
        sct_data[rank] = json.load(f)

def smooth(vals, w=50):
    out = []
    for i in range(len(vals)):
        start = max(0, i - w + 1)
        out.append(sum(vals[start:i+1]) / (i - start + 1))
    return out

# Color scheme: dense blue, SCT warm gradient
colors = {256: '#4CAF50', 128: '#FF9800', 64: '#FF5722', 32: '#E91E63'}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Panel 1: Loss curves ---
ax = axes[0]
d_steps = [l["step"] for l in dense_losses]
d_loss = [l["loss"] for l in dense_losses]
d_smooth = smooth(d_loss)
ax.plot(d_steps, d_loss, alpha=0.08, color='#2196F3')
ax.plot(d_steps, d_smooth, label='Dense (1.71B)', color='#2196F3', linewidth=2.5)

for rank in RANKS:
    s_steps = [l["step"] for l in sct_data[rank]]
    s_loss = [l["loss"] for l in sct_data[rank]]
    s_smooth = smooth(s_loss)
    ax.plot(s_steps, s_loss, alpha=0.08, color=colors[rank])
    ax.plot(s_steps, s_smooth, label=f'SCT r={rank}', color=colors[rank], linewidth=2)

ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Loss Convergence')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Panel 2: Perplexity curves ---
ax = axes[1]
d_ppl = [math.exp(min(l, 20)) for l in d_smooth]
ax.plot(d_steps, d_ppl, label='Dense (1.71B)', color='#2196F3', linewidth=2.5)

for rank in RANKS:
    s_steps = [l["step"] for l in sct_data[rank]]
    s_loss = [l["loss"] for l in sct_data[rank]]
    s_smooth = smooth(s_loss)
    s_ppl = [math.exp(min(l, 20)) for l in s_smooth]
    ax.plot(s_steps, s_ppl, label=f'SCT r={rank}', color=colors[rank], linewidth=2)

ax.set_xlabel('Step')
ax.set_ylabel('Perplexity')
ax.set_title('Perplexity Convergence')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# --- Panel 3: Compression vs Final PPL ---
ax = axes[2]
dense_final_ppl = math.exp(min(d_smooth[-1], 20))

compressions = []
final_ppls = []
for rank in sorted(RANKS):
    s_loss = [l["loss"] for l in sct_data[rank]]
    s_smooth_final = smooth(s_loss)[-1]
    final_ppl = math.exp(min(s_smooth_final, 20))
    # MLP compression: dense_layer / sct_layer
    dense_layer = 2048 * 5632
    sct_layer = 2048 * rank + rank + 5632 * rank
    comp = dense_layer / sct_layer
    compressions.append(comp)
    final_ppls.append(final_ppl)
    ax.scatter(comp, final_ppl, color=colors[rank], s=120, zorder=5)
    ax.annotate(f'r={rank}', (comp, final_ppl), textcoords='offset points',
                xytext=(8, 8), fontsize=10, color=colors[rank], fontweight='bold')

ax.axhline(y=dense_final_ppl, color='#2196F3', linestyle='--', linewidth=1.5,
           label=f'Dense PPL={dense_final_ppl:.1f}', alpha=0.8)
ax.plot(compressions, final_ppls, color='gray', linewidth=1, alpha=0.5, linestyle='--')
ax.set_xlabel('MLP Compression Ratio (×)')
ax.set_ylabel('Final Perplexity (smoothed)')
ax.set_title('Compression vs Quality')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# Footer
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
fig.text(0.5, 0.01,
         f'SmolLM2-1.7B | Alpaca | {gpu_name} | '
         f'Dense LR: {DENSE_LR} | SCT LR: {SCT_LR} | {STEPS} steps | EctoSpace/SCT',
         ha='center', fontsize=9, color='gray')

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('sct_rank_sweep_smollm2_1.7B.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved: sct_rank_sweep_smollm2_1.7B.png")

In [ ]:
# Cell 10: Summary table
import math
import json

with open("dense_losses.json") as f:
    dense_losses = json.load(f)

def smooth(vals, w=50):
    out = []
    for i in range(len(vals)):
        start = max(0, i - w + 1)
        out.append(sum(vals[start:i+1]) / (i - start + 1))
    return out

d_smooth = smooth([l["loss"] for l in dense_losses])
d_final = d_smooth[-1]
d_ppl = math.exp(min(d_final, 20))
d_time = sum(l["time"] for l in dense_losses) / 60
dense_params = 1_711_376_384

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

print(f"")
print(f"{'='*80}")
print(f"  SCT RANK SWEEP RESULTS: SmolLM2-1.7B on Alpaca")
print(f"  Hardware: {gpu_name}")
print(f"  Steps: {STEPS} | Dense LR: {DENSE_LR} | SCT LR: {SCT_LR} | Batch: {BATCH_SIZE}")
print(f"{'='*80}")
print(f"  {'Method':<14s} {'Params':>12s} {'Compression':>12s} {'Loss':>8s} {'PPL':>8s} {'GPU GB':>8s} {'Time':>8s} {'Step':>7s}")
print(f"  {'-'*14} {'-'*12} {'-'*12} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*7}")

d_step_time = sum(l["time"] for l in dense_losses) / len(dense_losses)
print(f"  {'Dense':<14s} {dense_params:>12,} {'1.0x':>12s} {d_final:>8.4f} {d_ppl:>8.2f} {'35.5':>8s} {d_time:>7.1f}m {d_step_time:>6.2f}s")

summary_rows = []
for rank in sorted(RANKS, reverse=True):
    with open(f"sct_r{rank}_losses.json") as f:
        sct_losses = json.load(f)
    s_smooth = smooth([l["loss"] for l in sct_losses])
    s_final = s_smooth[-1]
    s_ppl = math.exp(min(s_final, 20))
    s_time = sum(l["time"] for l in sct_losses) / 60
    s_step_time = sum(l["time"] for l in sct_losses) / len(sct_losses)

    meta = all_sct_meta[rank]
    params = meta["params"]
    gpu = meta["gpu_gb"]

    dense_layer = 2048 * 5632
    sct_layer = 2048 * rank + rank + 5632 * rank
    comp = dense_layer / sct_layer

    ppl_ratio = s_ppl / d_ppl
    label = f"SCT r={rank}"
    print(f"  {label:<14s} {params:>12,} {comp:>11.1f}x {s_final:>8.4f} {s_ppl:>8.2f} {gpu:>7.1f} {s_time:>7.1f}m {s_step_time:>6.2f}s")

    summary_rows.append({
        "rank": rank,
        "params": params,
        "compression": round(comp, 1),
        "final_loss_smoothed": round(s_final, 4),
        "final_ppl_smoothed": round(s_ppl, 2),
        "gpu_gb": round(gpu, 1),
        "time_min": round(s_time, 1),
        "step_time": round(s_step_time, 3),
        "ppl_ratio_vs_dense": round(ppl_ratio, 2)
    })

print(f"{'='*80}")

full_summary = {
    "model": "SmolLM2-1.7B",
    "dataset": "alpaca",
    "hardware": gpu_name,
    "steps": STEPS,
    "dense_lr": DENSE_LR,
    "sct_lr": SCT_LR,
    "batch_size": BATCH_SIZE,
    "dense": {
        "params": dense_params,
        "final_loss_smoothed": round(d_final, 4),
        "final_ppl_smoothed": round(d_ppl, 2),
        "time_min": round(d_time, 1)
    },
    "sct_ranks": summary_rows
}

with open("rank_sweep_summary.json", "w") as f:
    json.dump(full_summary, f, indent=2)
print(f"\nSummary saved: rank_sweep_summary.json")

In [ ]:
# Cell 11: Download results
from google.colab import files

files.download('sct_rank_sweep_smollm2_1.7B.png')
files.download('dense_losses.json')
for rank in RANKS:
    files.download(f'sct_r{rank}_losses.json')
files.download('rank_sweep_summary.json')
print("All files downloaded.")